In [ ]:
!pip install enoslib ipywidgets --break-system-packages

In [ ]:
!ssh rennes.grid5000.fr hostname

In [ ]:
!ssh-add ~/.ssh/id_ed25519

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [ ]:
from grid5000 import Grid5000
import enoslib as en
import logging
import os
from datetime import datetime, timedelta

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")  # type: ignore
gk = Grid5000.from_yaml(conf_file)

# map each cluster to its site
cluster_to_site = {}
for site in gk.sites.list():
    for cluster in site.clusters.list():
        cluster_to_site[cluster.uid] = site.uid

# JOB CONFIGURATION
JOB_NAME = "fcquic_local_eval"
# CLUSTER="vianden"
CLUSTER = "larochette"
JOB_WALLTIME = timedelta(hours=2, minutes=0)

# usage policy check: the job cannot cross the day to night boundary at 7pm if it was submitted before 5 pm. Any job started after 5 pm can cross the boundary
# I had the issue once so this check is there to avoid receiving a usage policy violation email...
datetime_now = datetime.now()
job_end_dt = datetime_now + JOB_WALLTIME
if (
    datetime_now.hour <= 17 and job_end_dt.hour >= 19 and datetime_now.weekday() <= 5
):  # only check during weekdays
    raise RuntimeError(
        "This job reservation will violate the usage policy and will cross the day night boundary"
    )

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()

conf = en.G5kConf.from_settings(
    job_name=JOB_NAME,
    walltime=str(JOB_WALLTIME),
    env_name="debian12-nfs",  # using debian13 here, was 12 before
    job_type=["deploy"],
).add_machine(
    roles=["server"],
    # servers=["larochette-4.luxembourg.grid5000.fr"],
    cluster=CLUSTER,
    nodes=1,
)

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

In [ ]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

# Fill in network information from nodes
roles = en.sync_info(roles, networks)

FRR_VERSION = "frr-10.4"

with en.actions(roles=roles, gather_facts=True) as a:
    a.apt(
        task_name="Install packages",
        name=[
            "tcpdump",
            "cmake",
            "clang",
            "python3.11-venv",
            "python-is-python3",
            "python3-pip",
            "btop",
            "htop",
        ],
        state="present",
    )

    # install frr
    a.file(
        task_name="Ensure apt keyring directory exists",
        path="/usr/share/keyrings",
        state="directory",
        mode="0755",
    )
    a.get_url(
        task_name="Download FRR GPG key",
        url="https://deb.frrouting.org/frr/keys.gpg",
        dest="/usr/share/keyrings/frrouting.gpg",
        mode="0644",
    )
    a.apt_repository(
        task_name="Add FRR apt repository",
        repo=f"deb [signed-by=/usr/share/keyrings/frrouting.gpg] https://deb.frrouting.org/frr bookworm {FRR_VERSION}",
        filename="frr",
        state="present",
    )
    a.apt(
        task_name="Install FRR packages",
        name=["frr", "frr-pythontools"],
        state="present",
        update_cache=True,
    )

    # rust
    a.get_url(
        task_name="Download rustup installer",
        url="https://sh.rustup.rs",
        dest="/tmp/rustup-init.sh",
        mode="0755",
    )
    a.shell(
        task_name="Install Rust stable",
        cmd="sh /tmp/rustup-init.sh -y --default-toolchain stable",
        creates="/root/.cargo/bin/rustup",
    )
    a.shell(
        task_name="Install Rust nightly toolchain",
        cmd="/root/.cargo/bin/rustup toolchain install nightly",
    )

    results = a.results

In [5]:
# print os and kernel versions
from enoslib.api import Results

res: Results = en.run_command("uname -a", roles=roles)
print([res.stdout for res in res])

Output()

Finished 1 tasks (uname -a) on {'larochette-5.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

['Linux larochette-5.luxembourg.grid5000.fr 6.1.0-44-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.1.164-1 (2026-03-09) x86_64 GNU/Linux']


### Uploading project with SCP 

In [6]:
import subprocess

local_bin_dir = f"/home/corentin/fcquic_applications_master_thesis"
remote_bin_dir = "/tmp/chat"

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"pushing binary to {host} (role: {role})")

        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/fcquic_chat"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/multicast-quic"], check=True
        )

        print("pushing chat app")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=logs/",
                "--exclude=baseline_logs/",
                "--exclude=tcp_logs/",
                "--exclude=tquic_logs/",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/fcquic_chat/",
                f"{host}:{remote_bin_dir}/fcquic_chat/",
            ],
            check=True,
        )
        print("pushing multicast quic dir")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/multicast-quic/",
                f"{host}:{remote_bin_dir}/multicast-quic/",
            ],
            check=True,
        )
        print("pushing evaluation dir")
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=grid5000/",
                "--exclude=venv/",
                "--exclude=results/",
                "--exclude=graphs/",
                f"{local_bin_dir}/evaluations/",
                f"{host}:{remote_bin_dir}/evaluations/",
            ],
            check=True,
        )

        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations/graphs/"], check=True
        )
print("pushed project to node")

pushing binary to larochette-5.luxembourg.grid5000.fr (role: server)


pushing chat app


pushing multicast quic dir


pushing evaluation dir


pushed project to node


#### Rsync script.npf

In [28]:
import subprocess

for role, nodes in roles.items():
    for node in nodes:
        host = node.address

        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=logs/",
                "--exclude=baseline_logs/",
                "--exclude=tcp_logs/",
                "--exclude=tquic_logs/",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/fcquic_chat/",
                f"{host}:{remote_bin_dir}/fcquic_chat/",
            ],
            check=True,
        )
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=target/",
                "--exclude=.git/",
                f"{local_bin_dir}/multicast-quic/",
                f"{host}:{remote_bin_dir}/multicast-quic/",
            ],
            check=True,
        )

        # subprocess.run(
        #     [
        #         "rsync",
        #         "-az",
        #         f"{local_bin_dir}/evaluations/topologies/configs/data/data.yaml",
        #         f"{host}:{remote_bin_dir}/evaluations/topologies/configs/data/data.yaml",
        #     ],
        #     check=True,
        # )
        # subprocess.run(
        #     [
        #         "rsync",
        #         "-az",
        #         f"{local_bin_dir}/evaluations/topologies/configs/latency/tiny_100mbps.yaml",
        #         f"{host}:{remote_bin_dir}/evaluations/topologies/configs/latency/tiny_100mbps.yaml",
        #     ],
        #     check=True,
        # )

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/tests/script.npf",
                f"{host}:{remote_bin_dir}/evaluations/tests/script.npf",
            ],
            check=True,
        )

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/run_test.sh",
                f"{host}:{remote_bin_dir}/evaluations/run_test.sh",
            ],
            check=True,
        )

### Installing deps 

In [7]:
en.run_command(
    f"pip install graphviz networkx pandas brokenaxes --break-system-packages",
    roles=roles,
)

Output()

Finished 1 tasks (pip install graphviz networkx pandas brokenaxes --break-system-packages) on
{'larochette-5.luxembourg.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[CommandResult(host='larochette-5.luxembourg.grid5000.fr', task='pip install graphviz networkx pandas brokenaxes --break-system-packages', status='OK', payload={'changed': True, 'stdout': "Collecting graphviz\n  Downloading graphviz-0.21-py3-none-any.whl (47 kB)\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 788.3 kB/s eta 0:00:00\nCollecting networkx\n  Downloading networkx-3.6.1-py3-none-any.whl (2.1 MB)\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.8 MB/s eta 0:00:00\nCollecting pandas\n  Downloading pandas-3.0.3-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.2 MB/s eta 0:00:00\nCollecting brokenaxes\n  Downloading brokenaxes-0.6.2-py3-none-any.whl (7.3 kB)\nCollecting numpy>=1.26.0\n  Downloading numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 144.9 MB/s eta 0:00:00\nRequirement already satisfied: python-dateutil>=2.8.2 in /usr/lib/python3/dist-packages (from pandas) (2.8.2)\nRequirement already satisfied: matplotlib>3.6 in /usr/lib/python3/dist-packages (from brokenaxes) (3.6.3)\nInstalling collected packages: numpy, networkx, graphviz, brokenaxes, pandas\n  Attempting uninstall: numpy\n    Found existing installation: numpy 1.24.2\n    Not uninstalling numpy at /usr/lib/python3/dist-packages, outside environment /usr\n    Can't uninstall 'numpy'. No files were found to uninstall.\nSuccessfully installed brokenaxes-0.6.2 graphviz-0.21 networkx-3.6.1 numpy-2.4.4 pandas-3.0.3", 'stderr': "ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.\nscipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.4.4 which is incompatible.\nWARNING: Running pip as the 'root' user can result in broken permissions and conflicting behaviour with the system package manager. It is recommended to use a virtual environment instead: https://pip.pypa.io/warnings/venv", 'rc': 0, 'cmd': 'pip install graphviz networkx pandas brokenaxes --break-system-packages', 'start': '2026-05-14 20:12:55.019009', 'end': '2026-05-14 20:13:01.651584', 'delta': '0:00:06.632575', 'msg': '', 'invocation': {'module_args': {'_raw_params': 'pip install graphviz networkx pandas brokenaxes --break-system-packages', '_uses_shell': True, 'expand_argument_vars': True, 'stdin_add_newline': True, 'strip_empty_ends': True, 'argv': None, 'chdir': None, 'executable': None, 'creates': None, 'removes': None, 'stdin': None}}, 'stdout_lines': ['Collecting graphviz', '  Downloading graphviz-0.21-py3-none-any.whl (47 kB)', '     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 788.3 kB/s eta 0:00:00', 'Collecting networkx', '  Downloading networkx-3.6.1-py3-none-any.whl (2.1 MB)', '     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.8 MB/s eta 0:00:00', 'Collecting pandas', '  Downloading pandas-3.0.3-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)', '     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.2 MB/s eta 0:00:00', 'Collecting brokenaxes', '  Downloading brokenaxes-0.6.2-py3-none-any.whl (7.3 kB)', 'Collecting numpy>=1.26.0', '  Downloading numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)', '     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 144.9 MB/s eta 0:00:00', 'Requirement already satisfied: python-dateutil>=2.8.2 in /usr/lib/python3/dist-packages (from pandas) (2.8.2)', 'Requirement already satisfied: matplotlib>3.6 in /usr/lib/python3/dist-packages (from brokenaxes) (3.6.3)', 'Installing collected packages: numpy, networkx, graphviz, brokenaxes, pandas', '  Attempting uninstall: numpy', '    Found existing installation: numpy 1.24.2', '    Not uninstalling numpy at /usr/lib/python3/dist-packages, outside environment /usr', "    Can't uninstall 'numpy'. No files 

*Important:* SSH into the machine `root@vianden-1.luxembourg.grid5000.fr`, then run the following commands:
- `cd /tmp/chat/evaluations`
- `python -m venv venv`
- `source ./venv/bin/activate`
- `pip install npf`

In [ ]:
en.run_command(f"cd {remote_bin_dir}/fcquic_chat && cargo build --release", roles=roles)

### Test setup

In [8]:
TEST_DIR_NAME = "latency"

# TOPO_CONF_NAME = "solo_10gbps"

TOPO_CONF_NAME = "tiny_100mbps"
# TOPO_CONF_NAME = "tiny_1000mbps"

# TOPO_CONF_NAME = "small_0%_loss_1000mbps"
# TOPO_CONF_NAME = "small_0_1%_loss_1000mbps"
# TOPO_CONF_NAME = "small_0_5%_loss_1000mbps"
# TOPO_CONF_NAME = "small_5%_loss_1000mbps"
# TOPO_CONF_NAME = "small_10%_loss_1000mbps"

# TOPO_CONF_NAME = "medium_0%_loss"
# TOPO_CONF_NAME = "medium_0_5%_loss"
# TOPO_CONF_NAME = "medium_1%_loss"
# TOPO_CONF_NAME = "medium_5%_loss"

# TEST_DIR_NAME = "receivers"
# TOPO_CONF_NAME = "receivers"

# TEST_DIR_NAME = "data"
# TOPO_CONF_NAME = "data"

USE_POISSON = "true"
print("command to run")
print(f"bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}")

command to run
bash run_test.sh latency tiny_100mbps true


### Running test script (NOTE: don't use this, it crashes the notebook)

In [ ]:
for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        subprocess.run(
            [
                "ssh",
                "root@" + host,
                f"cd {remote_bin_dir}/evaluations/ && bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}",
            ],
            check=True,
        )

### Collecting data

In [22]:
import subprocess
import os

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"

remote_out_dir = f"{remote_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"
local_out_dir = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"

os.makedirs(local_out_dir, exist_ok=True)

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"downloading results from {host}")
        subprocess.run(
            [
                "rsync",
                "-az",
                f"root@{host}:{remote_out_dir}{result_filename}.csv",
                f"root@{host}:{remote_out_dir}{result_filename}-TLOAD.csv",
                local_out_dir,
            ],
            check=True,
        )

print(f"results saved to {local_out_dir}{result_filename}")

downloading results from larochette-5.luxembourg.grid5000.fr


results saved to /home/corentin/fcquic_applications_master_thesis/evaluations/tests/latency/out/npf_out_tiny_100mbps_poisson


### Graph results

In [23]:
import subprocess

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"
graphs_dir = f"{local_bin_dir}/evaluations/graphs"
csv_path = (
    f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/{result_filename}.csv"
)
output_dir = f"{graphs_dir}/{TEST_DIR_NAME}"

os.makedirs(output_dir, exist_ok=True)

args = ["python", f"{graphs_dir}/{TEST_DIR_NAME}.py", csv_path, output_dir]
# if TEST_DIR_NAME == "latency":
#     args.append("--inset")

args.append("--no-title")
subprocess.run(
    args,
    check=True,
)

print(f"graphs written to {output_dir}")

is poisson?: True
ADDITIONAL_DATA_SIZE values: [np.int64(1100)]
Outlier threshold: 5875.810000000394
Baseline QUIC samples: 3580
Baseline TCP samples: 3526
Baseline TCP (NO TLS) samples: 3492
FC-QUIC samples: 873
FC-QUIC with FEC samples: 0
Tokio-quiche samples: 3381
Per run breakdown
QUIC:
  Run 4: 3580 samples
TCP:
  Run 1: 3526 samples
TCP_NO_TLS:
  Run 2: 3492 samples
FCQUIC:
  Run 0: 873 samples
TOKIO_QUICHE:
  Run 3: 3381 samples
---------------- FCQUIC or FCQUIC_FEC probably bugged during the test!! ----------------
min length of the dataframes: 0
graphs written to /home/corentin/fcquic_applications_master_thesis/evaluations/graphs/latency


### Compress results into a tarball

In [ ]:
# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"{local_out_dir}results.tar.gz",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"{local_out_dir}results.tar.gz",
        output_dir,
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

#### Uncompress tarball into results (opposite of cell above)

In [ ]:
# extract the tarball back into local_out_dir (inverse of the cell above)
local_out_dir = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"
output_dir = f"{graphs_dir}/{TEST_DIR_NAME}"
subprocess.run(
    [
        "cp",
        f"{output_dir}/results.tar.gz",
        local_out_dir,
    ],
    check=True,
)

subprocess.run(
    [
        "tar",
        "xzf",
        f"{local_out_dir}results.tar.gz",
        "-C",
        "/",
    ],
    check=True,
)

subprocess.run(
    [
        "rm",
        f"{local_out_dir}results.tar.gz",
    ],
    check=True,
)

### Stopping the current booking

In [29]:
provider.destroy()

INFO     [G5k] Reloading 280565 from luxembourg                          ]8;id=303875;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=898600;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#167\167]8;;\

INFO     [G5k] Killing the job (luxembourg, 280565)                      ]8;id=205088;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=406715;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#276\276]8;;\

INFO     [G5k] Job killed (luxembourg, 280565)                           ]8;id=110845;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=514499;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#257\257]8;;\